In [2]:
pip install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 32.1 MB/s  0:00:00m0:00:010:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
"""
Egalitarian Social Welfare Allocation
with Pareto Optimality and EF1 Analysis

University of Glasgow Student-Project Allocation Dataset
"""

import os
import glob
from pulp import*

# ============================================================
# INSERT YOUR DATA DIRECTORY PATH HERE
# ============================================================
DATA_DIR = "datasets/capacities/00038-00000003.dat"   # <-- change this
# ============================================================


# ------------------------------------------------------------------
# 1. PARSE FILES
# ------------------------------------------------------------------

def parse_preference_file(filepath):
    """
    Parse a Preflib .toc or .soi preference file.

    Voter line formats:
        .soi:  1: 20,18,19,21,22
        .toc:  1: 20,18,19,21,22,{1,3,4,5,...}

    Utility assignment:
        - Strictly ranked: utility = num_alternatives + 1 - rank_position
        - Tied/unranked in {}: utility = 1
        - Absent entirely (.soi): utility = 0
    """
    num_alternatives = 0
    project_list = []
    voter_lines = []

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('#'):
                if 'NUMBER ALTERNATIVES:' in line:
                    num_alternatives = int(line.split(':')[1].strip())
                elif 'ALTERNATIVE NAME' in line:
                    after_hash = line.lstrip('#').strip()
                    idx_str = after_hash.split(':')[0].split()[-1]
                    proj_id = str(int(idx_str) - 1)
                    project_list.append(proj_id)
            else:
                voter_lines.append(line)

    students = []
    utilities = {}
    student_id = 0

    for line in voter_lines:
        count_part, prefs_part = line.split(':', 1)
        count = int(count_part.strip())
        prefs_part = prefs_part.strip()

        tied_projects = []
        if '{' in prefs_part:
            before_tie = prefs_part[:prefs_part.index('{')].rstrip(',')
            tie_block  = prefs_part[prefs_part.index('{')+1 : prefs_part.rindex('}')]
            tied_projects = [str(int(p.strip()) - 1)
                             for p in tie_block.split(',') if p.strip()]
        else:
            before_tie = prefs_part

        strict_raw = [p.strip() for p in before_tie.split(',') if p.strip()]
        strict_projects = [str(int(p) - 1) for p in strict_raw]

        for _ in range(count):
            sid = f"s{student_id}"
            students.append(sid)
            utilities[sid] = {p: 0 for p in project_list}
            for rank_pos, proj in enumerate(strict_projects):
                utilities[sid][proj] = num_alternatives - rank_pos
            for proj in tied_projects:
                utilities[sid][proj] = 1
            student_id += 1

    return students, project_list, utilities


def parse_supervisor_file(filepath):
    """
    Parse supervisor .dat file.
    Format: Supervisor X, capacity, project_id
    """
    supervisors = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith('supervisor,'):
                continue
            parts = [p.strip() for p in line.split(',')]
            if len(parts) < 3:
                continue
            sup_id   = parts[0]
            capacity = int(parts[1])
            projects = parts[2].split()
            supervisors[sup_id] = {'capacity': capacity, 'projects': projects}
    return supervisors


def build_acceptable_pairs(students, projects, utilities):
    """Acceptable pairs: (student, project) where utility > 0."""
    return [(s, p) for s in students for p in projects
            if utilities[s].get(p, 0) > 0]


# ------------------------------------------------------------------
# 2. EGALITARIAN ILP
# ------------------------------------------------------------------

def solve_egalitarian(students, projects, supervisors, utilities, E):
    """
    Egalitarian ILP:
        max  z
        s.t. sum_{p} x_sp * w_sp  >= z    for all s in S   (min utility floor)
             sum_{p} x_sp          = 1    for all s in S   (one project each)
             sum_{s} x_sp         <= y_p  for all p in P   (project must be open)
             sum_{p in Pk} y_p    <= C_k  for all k in F   (supervisor capacity)
             x_sp, y_p in {0,1},  z >= 0
    """
    prob = LpProblem("Egalitarian_SWA", LpMaximize)

    x = {(s, p): LpVariable(f"x_{s}_{p}", cat='Binary') for (s, p) in E}
    y = {p: LpVariable(f"y_{p}", cat='Binary') for p in projects}
    z = LpVariable("z", lowBound=0)   # <-- the welfare floor we maximise

    # Objective
    prob += z, "Egalitarian_Welfare"

    # Each student gets exactly one project
    for s in students:
        prob += lpSum(x[s, p] for (s2, p) in E if s2 == s) == 1, \
               f"one_project_{s}"

    # Every student's utility >= z  (KEY egalitarian constraint)
    for s in students:
        prob += lpSum(x[s, p] * utilities[s][p] for (s2, p) in E if s2 == s) >= z, \
               f"min_utility_{s}"

    # Project only assigned if open
    for p in projects:
        prob += lpSum(x[s, p] for (s, p2) in E if p2 == p) <= y[p], \
               f"project_open_{p}"

    # Supervisor capacity
    for sup_id, sup_data in supervisors.items():
        sup_projs = [p for p in sup_data['projects'] if p in projects]
        if sup_projs:
            prob += lpSum(y[p] for p in sup_projs) <= sup_data['capacity'], \
                   f"supervisor_cap_{sup_id}"

    prob.solve(PULP_CBC_CMD(msg=0))

    allocation = {}
    if LpStatus[prob.status] == 'Optimal':
        for (s, p) in E:
            if value(x[s, p]) and value(x[s, p]) > 0.5:
                allocation[s] = p

    min_welfare   = value(z) if LpStatus[prob.status] == 'Optimal' else None
    total_welfare = sum(utilities[s][allocation[s]] for s in allocation)
    return allocation, min_welfare, total_welfare, LpStatus[prob.status]


# ------------------------------------------------------------------
# 3. PARETO OPTIMALITY CHECK
# ------------------------------------------------------------------

def check_pareto_optimality(allocation, students, utilities):
    """
    Check Pareto optimality by searching for a blocking swap pair (s1, s2):
    Both are weakly better off swapping projects, and at least one is strictly better.

    Returns: (is_pareto_optimal: bool, first_blocking_pair or None)
    """
    for s1 in students:
        for s2 in students:
            if s1 == s2:
                continue
            p1 = allocation.get(s1)
            p2 = allocation.get(s2)
            if not p1 or not p2:
                continue

            u_s1_now  = utilities[s1].get(p1, 0)
            u_s2_now  = utilities[s2].get(p2, 0)
            u_s1_swap = utilities[s1].get(p2, 0)
            u_s2_swap = utilities[s2].get(p1, 0)

            if (u_s1_swap >= u_s1_now and u_s2_swap >= u_s2_now and
                    (u_s1_swap > u_s1_now or u_s2_swap > u_s2_now)):
                return False, (s1, s2, p1, p2)

    return True, None


# ------------------------------------------------------------------
# 4. EF1 CHECK
# ------------------------------------------------------------------

def check_ef1(allocation, students, utilities):
    """
    Check Envy-Freeness up to One Good (EF1).

    Since each student receives exactly one project, EF1 and EF coincide:
    removing s2's single project leaves them with nothing, so s1 cannot
    envy s2 after that removal. This means EF1 is trivially satisfied in
    a one-item-per-agent allocation — but we still report raw envy pairs
    (cases where s1 strictly prefers s2's project over their own) as a
    meaningful fairness measure.

    Returns: (ef1_satisfied: bool, list of (s1, s2, u_own, u_envied) tuples)
    """
    violations = []
    for s1 in students:
        p1   = allocation.get(s1)
        u_s1 = utilities[s1].get(p1, 0) if p1 else 0
        for s2 in students:
            if s1 == s2:
                continue
            p2         = allocation.get(s2)
            u_s1_for_p2 = utilities[s1].get(p2, 0) if p2 else 0
            if u_s1_for_p2 > u_s1:
                violations.append((s1, s2, u_s1, u_s1_for_p2))

    return len(violations) == 0, violations


# ------------------------------------------------------------------
# 5. RESULTS PRINTER
# ------------------------------------------------------------------

def print_results(allocation, students, utilities, status,
                  total_welfare, min_welfare):
    print(f"\n{'='*55}")
    print(f"  EGALITARIAN ALLOCATION RESULTS")
    print(f"{'='*55}")
    print(f"  Solver status       : {status}")
    print(f"  Total welfare       : {total_welfare}")
    print(f"  Minimum welfare (z) : {min_welfare:.2f}")

    is_po, blocking = check_pareto_optimality(allocation, students, utilities)
    ef1_ok, violations = check_ef1(allocation, students, utilities)

    print(f"\n  Pareto Optimal : {'✓ Yes' if is_po else f'✗ No'}")
    if not is_po:
        s1, s2, p1, p2 = blocking
        print(f"    Blocking pair: {s1} and {s2} could swap "
              f"project {p1} ↔ {p2} and both be weakly better off")

    print(f"  EF1 Satisfied  : {'✓ Yes' if ef1_ok else f'✗ No'}")
    if not ef1_ok:
        print(f"    {len(violations)} envy pairs found "
              f"(students who prefer another student's project over their own)")
        print(f"    Note: in single-item allocation EF1 = EF, "
              f"so this also means EF is violated")

    print(f"\n  Assignments:")
    print(f"  {'Student':<10} {'Project':<12} {'Utility':<10} {'Rank'}")
    print(f"  {'-'*45}")
    for s, p in sorted(allocation.items()):
        u = utilities[s].get(p, 0)
        # Rank = position in student's preference (1 = top choice)
        sorted_prefs = sorted(utilities[s].items(), key=lambda x: -x[1])
        rank = next((i+1 for i, (proj, _) in enumerate(sorted_prefs)
                     if proj == p), '?')
        print(f"  {s:<10} {p:<12} {u:<10} {rank}")


# ------------------------------------------------------------------
# 6. MAIN
# ------------------------------------------------------------------

def main():
    pref_files = sorted(
        glob.glob(os.path.join(DATA_DIR, "*.toc")) +
        glob.glob(os.path.join(DATA_DIR, "*.soi"))
    )
    sup_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.dat")))

    if not pref_files:
        print(f"No preference files (.toc or .soi) found in: {DATA_DIR}")
        return

    for i, pref_file in enumerate(pref_files):
        sup_file = sup_files[i] if i < len(sup_files) else None
        year = os.path.basename(pref_file)
        print(f"\n{'#'*55}")
        print(f"  Processing: {year}")
        print(f"{'#'*55}")

        students, projects, utilities = parse_preference_file(pref_file)
        supervisors = parse_supervisor_file(sup_file) if sup_file else {}
        E = build_acceptable_pairs(students, projects, utilities)

        print(f"  Students: {len(students)} | "
              f"Projects: {len(projects)} | "
              f"Acceptable pairs: {len(E)}")

        allocation, min_w, total_w, status = solve_egalitarian(
            students, projects, supervisors, utilities, E)

        print_results(allocation, students, utilities,
                      status, total_w, min_w)


if __name__ == "__main__":
    main()

No preference files (.toc or .soi) found in: datasets/capacities/00038-00000003.dat
